# V-MD3 Continuous-Motion SAR Acquisition

Records continuous-motion SAR sweeps with the V-MD3 61 GHz FMCW radar mounted on a Galil-driven linear stage. Each sweep moves the stage across the aperture at a constant speed while the radar records, and logs the stage position for every frame.

**Hardware**

- V-MD3 radar on Ethernet (static IP `192.168.100.201`)
- Griffin Motion linear stage on a Galil DMC-30012 controller over USB (COM port)
- Stage scale is 125 nm per count with about 101 mm of travel between limit switches

**One-time install**

1. Install gclib from Galil's website using the `.exe` installer.
2. In PowerShell, install the Python wrapper:
   `py -m pip install https://www.galil.com/sw/pub/python/gclib-1.0.1-py3-none-any.whl`

Use wrapper version 1.0.1. The newer 2.0 wrapper needs Python 3.12 and has a different API.

**Each session**

1. Close GDK so the COM port is free.
2. Edit **Section 1 (Settings)** and run it.
3. Run **Section 2 (Helpers)** once.
4. Run **Section 3 (Start session)** to set up the radar network, connect the radar, and home the stage.

**Each sweep**

1. Set `FILE_NAME` in Settings and re-run it.
2. Run **Section 4 (Record a sweep)**.

**When finished**

- Run **Section 5 (End session)** to close both connections.
- **Section 6 (Export)** is optional and converts a `.bin` file to CSV.
- The **Appendix** holds one-time stage calibration checks. Only rerun them after changing the stage or limit switches.

**Output files** are saved to `DATA_DIR/SESSION_FOLDER/bins/` as `FILE_NAME.bin` plus `FILE_NAME_positions.csv`, with one summary row per sweep in `scan_log.csv`.

## 1. Settings

The only cell you normally edit. Re-run it after any change.

- **Output:** where files go. Change `FILE_NAME` before each sweep. Repeated names get `_2`, `_3`... instead of overwriting.
- **Sweep:** aperture length and frame count. The stage speed is set so `N_FRAMES` frames span `SWEEP_LENGTH_MM`. Keep the spacing under λ/4 (about 1.23 mm), which means at least 83 frames for 100 mm.
- **Radar:** changing `RADAR_MODE` or `RADAR_STREAMS` needs a reconnect, so re-run **3b** afterward.
- **Stage:** calibrated values for this stage. Only change them if the stage or limit switches change (see the Appendix), then re-run **3c**.

In [ ]:
# Output
DATA_DIR = "data"
SESSION_FOLDER = "continuous_motion_test"
FILE_NAME = "test_sweep"

# Sweep
SWEEP_LENGTH_MM = 100.0
N_FRAMES = 101
FRAME_PERIOD_S = None   # None measures it in 3b, or set e.g. 0.130

# Radar
RADAR_IP = "192.168.100.201"
RADAR_MODE = 0   # RSET 0 = 2D, 6 m max range, 4.69 cm range bins
RADAR_STREAMS = ["radc"]   # any of "radc", "rfft", "done"
RADAR_SETUP_SCRIPT = "C:/My Computer/vmd3/setup_radar.ps1"

# Stage
STAGE_PORT = "COM4 --baud 115200 -d"
COUNTS_PER_MM = 8_000   # 125 nm per count
STAGE_RANGE_COUNTS = 809_954   # travel between limit switches
HOME_DIRECTION = +1   # +1 homes to the right end, -1 to the left
HOME_BACKOFF_COUNTS = 1_000   # 0.125 mm off the limit switch
HOME_SPEED_COUNTS = 50_000   # counts/s, about 6 mm/s
SWEEP_ACCEL_COUNTS = 1_000_000   # counts/s², reaches sweep speed in under 0.1 s

# Derived values (no need to edit)
import os
SESSION_DIR = os.path.join(DATA_DIR, SESSION_FOLDER)
BIN_DIR = os.path.join(SESSION_DIR, "bins")
SESSION_LOG = os.path.join(SESSION_DIR, "scan_log.csv")
SWEEP_COUNTS = int(round(SWEEP_LENGTH_MM * COUNTS_PER_MM))

# Checks
VALID_STREAMS = ("radc", "rfft", "done")
RADAR_STREAMS = list(dict.fromkeys(s.strip().lower() for s in RADAR_STREAMS))
if not RADAR_STREAMS or any(s not in VALID_STREAMS for s in RADAR_STREAMS):
    raise ValueError(f"RADAR_STREAMS must use only {VALID_STREAMS}")
if not 0 <= RADAR_MODE <= 8:
    raise ValueError("RADAR_MODE must be 0 to 8")
if N_FRAMES < 2:
    raise ValueError("N_FRAMES must be at least 2")
if HOME_BACKOFF_COUNTS + SWEEP_COUNTS > STAGE_RANGE_COUNTS:
    raise ValueError("Sweep plus home backoff is longer than the stage travel")

print(f"[settings] Output : {os.path.join(BIN_DIR, FILE_NAME)}.bin")
print(f"[settings] Sweep  : {SWEEP_LENGTH_MM:g} mm, {N_FRAMES} frames")
print(f"[settings] Radar  : mode {RADAR_MODE}, streams {RADAR_STREAMS}")

## 2. Helpers

Run all three cells once per kernel, in order. No edits needed.

- **2a Radar:** network setup, connect and close, frame reading, frame period measurement
- **2b Stage:** controller commands, moves, homing, connect and close
- **2c Sweep and files:** the sweep recorder, position and session logs, and the `.bin` to CSV export

The session log columns changed in this version. Start a new `SESSION_FOLDER` (or delete the old `scan_log.csv`) so old and new rows don't mix.

### 2a Radar

In [ ]:
import os
import time
import socket
import base64
import ctypes
import subprocess
from pathlib import Path
import numpy as np
from lib.vmd3 import VMD3, RdotConfig

# Radar constants
RADAR_UDP_TIMEOUT_S = 1.0
RADAR_RCVBUF_BYTES = 8 * 1024 * 1024   # room to buffer frames while Python talks to the stage
STREAM_FLAGS = {"radc": RdotConfig.RADC, "rfft": RdotConfig.RFFT, "done": RdotConfig.DONE}
STREAM_HEADERS = {"radc": b"RADC", "rfft": b"RFFT", "done": b"DONE"}
CYCLE_ORDER = ["radc", "rfft", "done"]   # order streams arrive within one radar cycle

if "radar" not in globals():
    radar = None


def run_network_setup():
    """Run the radar network setup script as administrator and print its output."""
    script = Path(RADAR_SETUP_SCRIPT).resolve()
    log = Path(os.environ["TEMP"]) / "vmd3_setup_log.txt"
    if not script.is_file():
        raise FileNotFoundError(f"Setup script not found: {script}")
    log.unlink(missing_ok=True)

    # Run the script and capture everything it prints into the log
    script_q = str(script).replace("'", "''")
    log_q = str(log).replace("'", "''")
    inner = f"& '{script_q}' *>&1 | Out-File -FilePath '{log_q}' -Encoding utf8"
    encoded = base64.b64encode(inner.encode("utf-16-le")).decode()
    ps_args = ["-NoProfile", "-ExecutionPolicy", "Bypass", "-EncodedCommand", encoded]

    # Run directly if already elevated, otherwise go through a UAC prompt
    if ctypes.windll.shell32.IsUserAnAdmin():
        subprocess.run(["powershell", *ps_args])
    else:
        print("[radar] Approve the UAC prompt to run the network setup...")
        launcher = f"Start-Process powershell -Verb RunAs -Wait -WindowStyle Hidden -ArgumentList '{' '.join(ps_args)}'"
        subprocess.run(["powershell", "-NoProfile", "-Command", launcher])

    # Show the output and confirm it passed
    if not log.is_file():
        raise RuntimeError("No setup output. The UAC prompt may have been declined.")
    text = log.read_text(encoding="utf-8-sig")
    print(text)
    if "All checks passed" not in text:
        raise RuntimeError("Radar network setup did not pass. See the messages above.")
    print("[radar] Network ready.")


def close_sockets(obj):
    """Force-close an object's TCP and UDP sockets, ignoring errors."""
    for s in (getattr(obj, "sockTCP", None), getattr(obj, "sockUDP", None)):
        try:
            s.close()
        except Exception:
            pass


def radar_close():
    """Close the radar connection if one is open."""
    global radar
    if radar is None:
        return
    try:
        radar.sockTCP.settimeout(2.0)
        radar.disconnect()
        print("[radar] Disconnected.")
    except Exception as e:
        print(f"[radar] Close warning: {e}. Sockets force-closed.")
        close_sockets(radar)
    radar = None


def radar_connect(attempts=3):
    """Open and configure the radar, retrying if the handshake stalls."""
    global radar
    if radar is not None:
        radar_close()
        time.sleep(2)   # let the radar release the old session
    for i in range(attempts):
        r = VMD3(tcp_ip=RADAR_IP)
        try:
            r.connect()
            r.sockTCP.settimeout(5.0)
            r.set_rset_config(RADAR_MODE)
            r.set_output_config([STREAM_FLAGS[s] for s in RADAR_STREAMS])
        except OSError as e:
            print(f"[radar] Handshake stalled ({e or 'timeout'}), retry {i + 1}/{attempts}...")
            close_sockets(r)
            time.sleep(2)
            continue
        r.sockUDP.settimeout(RADAR_UDP_TIMEOUT_S)
        r.sockUDP.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, RADAR_RCVBUF_BYTES)
        radar = r
        print(f"[radar] Ready. Mode {RADAR_MODE}, streams {RADAR_STREAMS}")
        return
    raise RuntimeError("Radar did not respond. Power cycle it, wait about 10 s, then re-run 3a and 3b.")


def read_frame(headers):
    """Read the next complete frame whose header is in headers."""
    while True:
        data = radar.recv_udp()
        if data[:4] not in headers:
            continue
        length = int.from_bytes(data[4:8], "little")
        while len(data) < length + 8:
            data += radar.recv_udp()
        if len(data) == length + 8:
            return data


def drain_udp():
    """Discard everything already waiting in the UDP buffer."""
    radar.sockUDP.settimeout(0.05)
    try:
        while True:
            radar.recv_udp()
    except socket.timeout:
        pass
    finally:
        radar.sockUDP.settimeout(RADAR_UDP_TIMEOUT_S)


def wait_for_frame(header):
    """Block until the next complete frame with this header arrives."""
    while True:
        try:
            return read_frame((header,))
        except socket.timeout:
            pass


def sync_stream():
    """Last selected stream in each radar cycle, used to mark cycle boundaries."""
    return [s for s in CYCLE_ORDER if s in RADAR_STREAMS][-1]


def measure_frame_period(n_cycles=30):
    """Time n radar cycles and return the median cycle period in seconds."""
    header = STREAM_HEADERS[sync_stream()]
    drain_udp()
    stamps = []
    while len(stamps) < n_cycles + 1:
        wait_for_frame(header)
        stamps.append(time.perf_counter())
    return float(np.median(np.diff(stamps)))


print("[helpers] Radar helpers loaded.")

### 2b Stage

In [ ]:
import gclib

# Stage constants
STAGE_AXIS = "A"

if "stage" not in globals():
    stage = None


def stage_reconnect(attempts=3):
    """Close and reopen the controller connection to clear out-of-sync replies."""
    try:
        stage.GClose()
    except gclib.GclibError:
        pass
    for i in range(attempts):
        time.sleep(0.5)
        try:
            stage.GOpen(STAGE_PORT)
            return
        except gclib.GclibError:
            if i == attempts - 1:
                raise


def stage_cmd(cmd):
    """Send one command, reconnecting once after a timeout, and return the reply."""
    try:
        return stage.GCommand(cmd).strip()
    except gclib.GclibError as e:
        if "timed out" not in str(e):
            raise RuntimeError(f"Stage rejected '{cmd}': {stage.GCommand('TC1').strip()}") from e
        print(f"[stage] '{cmd}' timed out, reconnecting...")
        stage_reconnect()
        return stage.GCommand(cmd).strip()


def stage_state():
    """Return (position in counts, True if the axis is moving)."""
    pos = float(stage_cmd(f"MG _TP{STAGE_AXIS}"))
    moving = float(stage_cmd(f"MG _BG{STAGE_AXIS}")) > 0.5
    return int(pos), moving


def stage_wait(timeout_s=60):
    """Poll until the axis stops, stopping it if it takes too long."""
    t_end = time.perf_counter() + timeout_s
    while stage_state()[1]:
        if time.perf_counter() > t_end:
            stage_cmd(f"ST{STAGE_AXIS}")
            raise TimeoutError("Stage move did not finish in time.")
        time.sleep(0.05)


def stage_move(counts, speed, absolute=False, timeout_s=60):
    """Move at the given speed and block until the move finishes."""
    stage_cmd(f"SP{STAGE_AXIS}={int(speed)}")
    stage_cmd(f"{'PA' if absolute else 'PR'}{STAGE_AXIS}={int(counts)}")
    stage_cmd(f"BG{STAGE_AXIS}")
    try:
        stage_wait(timeout_s)
    except BaseException:
        stage_cmd(f"ST{STAGE_AXIS}")
        raise


def stage_seek_end(direction, speed, timeout_s=60):
    """Jog toward an end until a limit switch or stall stops it."""
    stage_cmd(f"JG{STAGE_AXIS}={int(direction * speed)}")
    try:
        stage_cmd(f"BG{STAGE_AXIS}")
    except RuntimeError as e:
        if "limit switch" in str(e):
            return   # already sitting on that limit switch
        raise
    t_end = time.perf_counter() + timeout_s
    last_pos, last_t = stage_state()[0], time.perf_counter()
    try:
        while True:
            time.sleep(0.05)
            pos, moving = stage_state()
            if not moving:
                break   # the limit switch stopped it
            if abs(pos - last_pos) > 100:
                last_pos, last_t = pos, time.perf_counter()
            elif time.perf_counter() - last_t > 0.3:
                break   # pressed against the end stop
            if time.perf_counter() > t_end:
                raise TimeoutError("Stage never reached the end.")
    finally:
        stage_cmd(f"ST{STAGE_AXIS}")
        stage_wait(10)


def stage_home():
    """Seek the home-side limit switch, back off, and set that position as zero."""
    stage_cmd(f"SH{STAGE_AXIS}")
    stage_cmd(f"AC{STAGE_AXIS}={SWEEP_ACCEL_COUNTS}")
    stage_cmd(f"DC{STAGE_AXIS}={SWEEP_ACCEL_COUNTS}")
    stage_seek_end(HOME_DIRECTION, HOME_SPEED_COUNTS)
    stage_cmd(f"SH{STAGE_AXIS}")   # re-enable in case the end tripped the error limit
    stage_move(-HOME_DIRECTION * HOME_BACKOFF_COUNTS, HOME_SPEED_COUNTS)
    time.sleep(0.2)
    stage_cmd(f"DP{STAGE_AXIS}=0")


def stage_go_to_start():
    """Return to the zero position set by homing and let it settle."""
    stage_move(0, HOME_SPEED_COUNTS, absolute=True)
    time.sleep(0.2)


def stage_close():
    """Stop the stage and close the controller connection if one is open."""
    global stage
    if stage is None:
        return
    try:
        stage_cmd(f"ST{STAGE_AXIS}")
        stage.GClose()
        print("[stage] Disconnected.")
    except Exception as e:
        print(f"[stage] Close warning: {e}")
    stage = None


def stage_connect():
    """Open the controller and stop any leftover motion or GDK program."""
    global stage
    stage_close()
    stage = gclib.py()
    stage.GOpen(STAGE_PORT)
    print(f"[stage] Connected: {stage.GInfo()}")
    stage_cmd(f"ST{STAGE_AXIS}")
    stage_wait()
    try:
        stage_cmd("HX")   # halt any program left running from GDK
    except RuntimeError:
        pass


print("[helpers] Stage helpers loaded.")

### 2C Sweep & Files

In [ ]:
import csv
from datetime import datetime
import pandas as pd

# Sweep constants
WAVELENGTH_MM = 299_792_458 / 61.0e9 * 1e3
LOG_HEADER = ["file", "start", "stop", "duration_s", "n_frames",
              "radar_mode", "streams", "sweep_mm", "speed_mm_s"]
POSITION_HEADER = ["record", "stream", "stream_frame", "t_s", "galil_counts",
                   "x_mm", "x_center_mm", "moving"]


def sweep_speed(period):
    """Stage speed in counts/s that spreads N_FRAMES frames over the sweep."""
    return int(round(SWEEP_COUNTS / ((N_FRAMES - 1) * period)))


def print_sweep_plan(period):
    """Print the frame rate, sweep speed, and frame spacing for the current settings."""
    speed = sweep_speed(period)
    spacing = SWEEP_LENGTH_MM / (N_FRAMES - 1)
    print(f"[sweep] Frame period  : {period * 1e3:.1f} ms ({1 / period:.2f} Hz)")
    print(f"[sweep] Sweep speed   : {speed} counts/s ({speed / COUNTS_PER_MM:.2f} mm/s)")
    print(f"[sweep] Sweep time    : {SWEEP_COUNTS / speed:.1f} s for {N_FRAMES} frames")
    print(f"[sweep] Frame spacing : {spacing:.3f} mm (λ/4 = {WAVELENGTH_MM / 4:.3f} mm)")
    if spacing > WAVELENGTH_MM / 4:
        min_frames = int(np.ceil(SWEEP_LENGTH_MM / (WAVELENGTH_MM / 4))) + 1
        print(f"[sweep] Warning: spacing exceeds λ/4. Use N_FRAMES ≥ {min_frames} to avoid grating lobes.")


def next_file_path():
    """Path for FILE_NAME in BIN_DIR, adding _2, _3... instead of overwriting."""
    path = os.path.join(BIN_DIR, f"{FILE_NAME}.bin")
    i = 2
    while os.path.exists(path):
        path = os.path.join(BIN_DIR, f"{FILE_NAME}_{i}.bin")
        i += 1
    return path


def ensure_session_files():
    """Create the session folders and the session log with its header if missing."""
    os.makedirs(BIN_DIR, exist_ok=True)
    if not os.path.exists(SESSION_LOG):
        with open(SESSION_LOG, "w", newline="") as f:
            csv.writer(f).writerow(LOG_HEADER)


def save_positions(path, rows):
    """Write the per-record position log for one sweep."""
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(POSITION_HEADER)
        writer.writerows(rows)


def append_log(name, start, stop, n_frames, speed):
    """Add one summary row for a sweep to the session log."""
    with open(SESSION_LOG, "a", newline="") as f:
        csv.writer(f).writerow([
            name, start.isoformat(timespec="seconds"), stop.isoformat(timespec="seconds"),
            f"{(stop - start).total_seconds():.2f}", n_frames, RADAR_MODE,
            "+".join(RADAR_STREAMS), f"{SWEEP_LENGTH_MM:g}", f"{speed / COUNTS_PER_MM:.3f}",
        ])


def print_summary(bin_path, pos_path, rows, duration_s):
    """Print the frame count, aperture span, and any dropped-frame gaps."""
    first = [s for s in CYCLE_ORDER if s in RADAR_STREAMS][0]
    xc = np.array([float(r[6]) for r in rows if r[1] == first])
    gaps = np.abs(np.diff(xc))
    print(f"[sweep] Saved {os.path.basename(bin_path)}: {len(rows)} records, "
          f"{os.path.getsize(bin_path) / 1e6:.2f} MB, {duration_s:.1f} s")
    print(f"[sweep] {len(xc)} {first.upper()} frames (target {N_FRAMES}), center position "
          f"{xc[0]:+.2f} to {xc[-1]:+.2f} mm, mean spacing {gaps.mean():.3f} mm")
    big = np.where(gaps > 1.5 * SWEEP_LENGTH_MM / (N_FRAMES - 1))[0]
    if big.size:
        print(f"[sweep] Warning: {big.size} dropped frame gap(s), largest {gaps.max():.2f} mm "
              f"near center {xc[big[0]]:+.2f} mm. Consider re-running this sweep.")
    print(f"[sweep] Positions → {os.path.basename(pos_path)}")


def record_sweep():
    """Run one sweep, record every frame while the stage moves, and save the results."""
    period = FRAME_PERIOD_S or globals().get("frame_period")
    if period is None:
        raise RuntimeError("No frame period. Run 3b or set FRAME_PERIOD_S in Settings.")
    speed = sweep_speed(period)
    timeout_s = SWEEP_COUNTS / speed + 10
    half_mm = SWEEP_LENGTH_MM / 2

    # Output files
    ensure_session_files()
    bin_path = next_file_path()
    pos_path = bin_path[:-4] + "_positions.csv"

    # Stream bookkeeping
    accepted = tuple(STREAM_HEADERS[s] for s in RADAR_STREAMS)
    header_to_stream = {STREAM_HEADERS[s]: s for s in RADAR_STREAMS}
    sync = sync_stream()
    counts = {s: 0 for s in RADAR_STREAMS}
    rows = []

    # Park at the start and load the sweep move without starting it
    print("[sweep] Moving to start...")
    stage_go_to_start()
    stage_cmd(f"SP{STAGE_AXIS}={speed}")
    stage_cmd(f"AC{STAGE_AXIS}={SWEEP_ACCEL_COUNTS}")
    stage_cmd(f"DC{STAGE_AXIS}={SWEEP_ACCEL_COUNTS}")
    stage_cmd(f"PR{STAGE_AXIS}={-HOME_DIRECTION * SWEEP_COUNTS}")

    # Start the move right after a radar cycle ends
    drain_udp()
    wait_for_frame(STREAM_HEADERS[sync])
    stage_cmd(f"BG{STAGE_AXIS}")
    t0 = time.perf_counter()
    start = datetime.now()
    print(f"[sweep] {SWEEP_LENGTH_MM:g} mm at {speed / COUNTS_PER_MM:.2f} mm/s, "
          f"target {N_FRAMES} frames → {os.path.basename(bin_path)}")

    # Record until the stage stops and the current cycle is complete
    with open(bin_path, "wb") as out:
        try:
            while True:
                if time.perf_counter() - t0 > timeout_s:
                    raise TimeoutError("Sweep took too long. Check the radar stream and the stage.")
                try:
                    frame = read_frame(accepted)
                except socket.timeout:
                    continue
                t = time.perf_counter() - t0
                pos, moving = stage_state()
                stream = header_to_stream[frame[:4]]
                x_mm = -HOME_DIRECTION * pos / COUNTS_PER_MM

                out.write(frame)
                rows.append([len(rows), stream, counts[stream], f"{t:.4f}", pos,
                             f"{x_mm:.4f}", f"{half_mm - x_mm:.4f}", int(moving)])
                counts[stream] += 1

                if stream == sync:
                    print(f"   cycle {counts[stream]:3d}   x = {x_mm:7.2f} mm   "
                          f"center {half_mm - x_mm:+7.2f} mm   {'moving' if moving else 'stopped'}")
                    if not moving:
                        break
        except BaseException:
            stage_cmd(f"ST{STAGE_AXIS}")
            raise
    stop = datetime.now()

    # Save logs and report
    save_positions(pos_path, rows)
    append_log(os.path.basename(bin_path), start, stop, counts[sync], speed)
    print_summary(bin_path, pos_path, rows, (stop - start).total_seconds())


def read_records(bin_path):
    """Yield (stream, payload) for each record in a .bin file, in file order."""
    names = {v: k for k, v in STREAM_HEADERS.items()}
    with open(bin_path, "rb") as f:
        while header := f.read(8):
            length = int.from_bytes(header[4:8], "little")
            yield names.get(header[:4], "unknown"), f.read(length)


def decode_radc(payload):
    """Decode a 2D RADC payload to (sample, chirp, rx) complex samples."""
    raw = np.frombuffer(payload, dtype="<i2").reshape(64, 4, 256)
    iq = raw[:, :, 1::2] + 1j * raw[:, :, 0::2]
    return np.transpose(iq, (2, 0, 1))


def decode_rfft(payload):
    """Decode a 2D RFFT payload to (range_bin, chirp, rx) complex samples."""
    raw = np.frombuffer(payload, dtype="<i2").reshape(4, 128, 64, 2)
    iq = raw[..., 1] + 1j * raw[..., 0]
    return np.transpose(iq, (1, 2, 0))


def export_long_csv(bin_path, streams=("radc",)):
    """Write one row per (frame, bin, chirp, channel) I/Q value to a CSV next to the .bin."""
    decoders = {"radc": decode_radc, "rfft": decode_rfft}
    counts = {s: 0 for s in streams}
    parts = []
    for stream, payload in read_records(bin_path):
        if stream not in streams or stream not in decoders:
            continue
        cube = decoders[stream](payload)
        b, c, h = np.meshgrid(*map(np.arange, cube.shape), indexing="ij")
        parts.append(pd.DataFrame({
            "stream": stream, "frame": counts[stream], "bin": b.ravel(), "chirp": c.ravel(),
            "channel": h.ravel(), "I": cube.real.ravel(), "Q": cube.imag.ravel(),
        }))
        counts[stream] += 1
    if not parts:
        print(f"[export] No {list(streams)} frames found in {os.path.basename(bin_path)}")
        return
    out_path = os.path.splitext(bin_path)[0] + "_long.csv"
    df = pd.concat(parts, ignore_index=True)
    df.to_csv(out_path, index=False)
    print(f"[export] Wrote {os.path.basename(out_path)}: {len(df):,} rows {counts}")


print("[helpers] Sweep and file helpers loaded.")

## 3. Start session

Run these at the start of each session, after Sections 1 and 2.

- **3a** only after plugging in or power cycling the radar. It opens a UAC prompt.
- **3b** connects the radar and measures its frame period. Re-run it after changing `RADAR_MODE` or `RADAR_STREAMS`. If it hangs after "Connected to UDP", power cycle the radar, wait about 10 s, then run 3a and 3b again.
- **3c** connects the stage and homes it to the right limit switch. Close GDK first. Re-run it if the stage is bumped or power cycled.

### 3a. Radar network setup

In [ ]:
run_network_setup()

### 3b. Connect radar and measure frame period

In [ ]:
radar_connect()
frame_period = FRAME_PERIOD_S or measure_frame_period()
print_sweep_plan(frame_period)

### 3c. Connect and home stage

In [ ]:
stage_connect()
print("[stage] Homing...")
stage_home()
print(f"[stage] Homed. Position {stage_state()[0]} counts = start of sweep.")

## 4. Record a sweep

Repeat for each sweep:

1. Set `FILE_NAME` in Settings and re-run Settings.
2. Run the cell below.

The stage returns to the start, waits for the end of a radar cycle, then sweeps the full aperture at constant speed while every frame is recorded. Recording stops once the stage stops and that frame finishes.

**Saved to `BIN_DIR`:**

- `FILE_NAME.bin` holds the raw radar frames in recording order.
- `FILE_NAME_positions.csv` has one row per frame:
  - `stream`, `stream_frame`: which frame in the `.bin` (e.g. RADC frame 12)
  - `t_s`: arrival time since motion start
  - `galil_counts`: raw encoder position
  - `x_mm`: 0 at the start (right) end, increasing toward the left
  - `x_center_mm`: position from the sweep center, + at the right end, − at the left end
  - `moving`: 1 while the stage was moving

A summary row is also added to `scan_log.csv`.

Use the logged `x_center_mm` positions for SAR processing rather than assuming exact 1 mm spacing. If the summary warns about a dropped-frame gap, re-run the sweep.

In [ ]:
record_sweep()

## 5. End session

Stops the stage and closes both connections. Run this before restarting the kernel or closing VS Code so the radar sees a clean disconnect, which helps avoid the connection hang in 3b next time.

Safe to run even if nothing is connected.

In [ ]:
radar_close()
stage_close()

## 6. Export to CSV (optional)

Converts a saved `.bin` into a long-format CSV with one row per (frame, bin, chirp, channel) value and separate `I` and `Q` columns. The CSV is saved next to the `.bin` with `_long.csv` added.

- `EXPORT_FILE` defaults to the current `FILE_NAME`. Set it to any other `.bin` in `BIN_DIR` to export that one instead, such as `"ball_sweep_2.bin"`.
- `EXPORT_STREAMS` can include `"radc"` and `"rfft"`, but only streams that were recorded get exported.
- 2D mode (`RADAR_MODE = 0`) only.

This doesn't need the radar or stage connected, just Settings and Section 2.

Long format is large. Each RADC or RFFT frame is 32,768 rows, so a 101-frame RADC sweep is about 3.3 million rows.

In [ ]:
EXPORT_FILE = f"{FILE_NAME}.bin"
EXPORT_STREAMS = ["radc"]

export_long_csv(os.path.join(BIN_DIR, EXPORT_FILE), EXPORT_STREAMS)